# 15.13 如何公平比較 Dense／MoE？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：四位師傅都在，這張訂單只請兩位動手**

例中只派給A與C；未選中師傅的權重仍佔儲存空間。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/moe.svg")))

**先想一想：**Dense與MoE hidden相同，總參數就相同嗎？

比較需要先選一種公平：存一樣多參數，或每token做近似一樣的計算。兩種match往往導致不同寬度，結果要分兩張表。

**把直覺寫成數學：**params-matched與compute-matched是不同實驗；另匹配資料、tokens與seed。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.modern import DenseFFN, MoEFFN

for name, layer in (("Dense", DenseFFN(8, hidden=32)), ("MoE", MoEFFN(8, experts=4, top_k=2, hidden=8))):
    print(name, sum(p.numel() for p in layer.parameters()))
print("另搜hidden以匹配總參數；記錄差距，品質待GPU訓練")

**如何讀結果：**這裏只比較參數候選，不宣稱已經compute-matched；router與共享attention也要列。

**只改一件事：**只把top_k改成1，觀察總參數不變、啟動計算變。
